# 남해안 3개 여행지 검색 관심도 트렌드 분석

- 데이터: 네이버 데이터랩 검색어트렌드(주간, 상대 지수 0~100), `collect.py`로 수집
- 지역: 통영 / 거제 / 남해 (각각 `○○ 여행` + `○○여행` 합산)
- 이 노트북은 **API를 호출하지 않는다**. 저장된 `data/search_trend_weekly.csv`만 읽는다.
- 실행: 위에서부터 순서대로 (Cursor 상단 `Run All` 또는 셀마다 `Shift+Enter`)

## 단계 4-1. 데이터 불러오기

CSV의 날짜 열은 글자로 저장돼 있다. `parse_dates`로 **날짜 자료형**으로 바꿔 읽어야
"7일 간격인가", "몇 월인가" 같은 날짜 계산을 할 수 있다.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "data/search_trend_weekly.csv",
    index_col="period",      # 날짜 열을 표의 기준(행 이름)으로 사용
    parse_dates=["period"],  # 글자 '2021-01-04' → 날짜 자료형
    encoding="utf-8-sig",
)
df.head()

## 단계 4-2. 기본 정보 확인 (기간 · 컬럼 · 결측치)

과제 요구사항 Ⅱ-3의 세 가지를 차례로 확인한다.

In [ ]:
print("행 수(주):", len(df), " / 열:", list(df.columns))
print("기간:", df.index.min().date(), "~", df.index.max().date())
print()
print("[자료형] 숫자(float)여야 계산 가능")
print(df.dtypes)
print()
print("[결측치] 지역별 빈칸 수")
print(df.isna().sum())

### 날짜 빠짐 확인

결측치는 **빈칸**만 있는 게 아니다. 어떤 주가 **통째로 빠져 있으면** 빈칸조차 생기지 않는다.
그래서 이웃한 날짜끼리의 간격이 전부 7일인지 따로 확인한다.

In [ ]:
gaps = df.index.to_series().diff().dropna()   # 바로 앞 날짜와의 차이
print("날짜 간격 종류:", gaps.value_counts().to_dict())
print("모든 날짜가 월요일인가:", (df.index.dayofweek == 0).all())

### 기초 통계

- `mean`(평균)과 `50%`(중앙값)의 차이가 크면, 일부 큰 값이 평균을 끌어올리고 있다는 신호다.
- `std`(표준편차)는 값이 평균에서 평소 얼마나 벗어나는지를 나타낸다.

In [ ]:
df.describe().round(1)

## 단계 4-3. 경계 주 처리

- 데이터랩은 주간 값을 **월요일 시작 주**로 묶는다. 그래서 첫 주(2020-12-28)와 마지막 주(2025-12-29)는 요청 기간(2021-01-01~2025-12-31)과 일부만 겹친다.
- **결정: 262주 모두 유지.** 원본을 그대로 쓰고, 두 주의 값이 앞뒤 주와 비슷해 분석을 왜곡하지 않는다.
- 연도·월별로 묶을 때는 **주의 시작일(월요일)** 기준으로 분류한다. 예: 2020-12-28 주는 2020년 12월로 들어간다.

In [ ]:
df.iloc[[0, 1, -2, -1]]   # 처음 2주와 마지막 2주를 나란히 비교

## 단계 4-4. 이상치 탐지 (IQR 기준)

**IQR(사분위 범위)**: 값을 작은 순으로 줄 세웠을 때 가운데 50%가 차지하는 폭.
- Q1 = 하위 25% 지점, Q3 = 상위 25% 지점, IQR = Q3 − Q1
- **Q3 + 1.5×IQR 보다 크거나 Q1 − 1.5×IQR 보다 작으면** 이상치 후보로 본다.

평균·표준편차 대신 IQR을 쓰는 이유: 평균과 표준편차는 큰 값 자체에 끌려가 기준선이 같이 올라간다.
IQR은 가운데 절반만 보고 기준을 정해서 튀는 값의 영향을 덜 받는다.

In [ ]:
q1 = df.quantile(0.25)
q3 = df.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

pd.DataFrame({"하한": lower, "상한": upper}).round(1)

In [ ]:
# True/False 표: 해당 주·지역 값이 기준선을 벗어나면 True
is_outlier = (df > upper) | (df < lower)
print("지역별 이상치 후보 수:")
print(is_outlier.sum())

### 이상치 후보가 '오류'인가 '실제 사건'인가

값을 지우거나 고치기 전에, 후보들이 **언제** 몰려 있는지 본다.
특정 달에 매년 반복된다면 데이터 오류가 아니라 계절적인 실제 관심 급증일 가능성이 크다.

In [ ]:
# 이상치 후보를 (날짜, 지역, 값) 목록으로 펼친 뒤 월별로 개수 세기
outliers = df[is_outlier].stack().rename("ratio").reset_index()
outliers.columns = ["period", "region", "ratio"]
outliers["year"] = outliers["period"].dt.year
outliers["month"] = outliers["period"].dt.month

pd.crosstab(outliers["region"], outliers["month"])   # 행=지역, 열=월, 값=후보 개수

In [ ]:
pd.crosstab(outliers["region"], outliers["year"])    # 연도별로는 어떻게 흩어져 있나

### 처리 결정: 표시만 하고 값은 유지

- 판단 근거와 결론은 위 두 표의 결과를 보고 README 단계 4 기록에 **본인 말로** 적는다.
- 처리 방식: 값은 바꾸지 않는다. 이후 분석에서 쓸 수 있도록 "이상치 후보였는가"만 표시해 둔다.
- 버린 선택지: 제거(휴가철 데이터가 사라져 Q1·Q3의 답이 없어짐), 상한선으로 깎기(여름 최고점이 낮아져 계절성이 실제보다 약하게 보임)

In [ ]:
# 원래 값은 그대로 두고, 지역별 '이상치 후보 여부' 열만 옆에 붙인 사본을 만든다
flags = is_outlier.add_suffix("_이상치후보")
df_checked = df.join(flags)
df_checked[df_checked.filter(like="_이상치후보").any(axis=1)].head(10)

---
# 단계 5. 시계열 기법 ① 이동평균 + 시각화

**이동평균(moving average)**: 어떤 주의 값을 그 주 혼자가 아니라 **앞뒤 몇 주를 묶은 평균**으로 바꿔 보는 방법.
주 단위 값은 한 주만 특별한 일이 있어도 크게 흔들린다(노이즈). 이 흔들림을 눌러
"큰 흐름(추세)"이 보이게 만드는 것이 목적이다.

- 창(window) 12주 = 약 3개월. 계절이 바뀌는 흐름은 남기고, 한두 주짜리 요동은 지운다.
- `center=True`: 그 주를 **가운데** 두고 앞 6주·뒤 6주를 평균한다. 앞 12주만 쓰면 그래프가 실제보다 오른쪽으로 밀려 보인다.

## 5-1. 한글 폰트 설정

matplotlib은 기본 글꼴에 한글이 없어 축·범례의 한글이 네모(□)로 나온다.
컴퓨터에 있는 한글 글꼴을 지정해 두면 해결된다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 설치된 글꼴 중 한글 글꼴을 순서대로 찾아 첫 번째로 있는 것을 쓴다
installed = {f.name for f in font_manager.fontManager.ttflist}
for name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if name in installed:
        matplotlib.rcParams["font.family"] = name
        break
matplotlib.rcParams["axes.unicode_minus"] = False  # 음수 기호가 깨지는 것 방지

print("사용 글꼴:", matplotlib.rcParams["font.family"])

## 5-2. 12주 이동평균 계산

`rolling(12)`은 "12주짜리 창을 한 주씩 밀면서 본다"는 뜻이고, `.mean()`이 그 창 안의 평균을 낸다.

In [ ]:
ma12 = df.rolling(window=12, center=True, min_periods=6).mean()

# 원본과 이동평균을 나란히 확인 (2022년 여름 앞뒤)
비교 = df.join(ma12.add_suffix("_12주평균")).loc["2022-07-04":"2022-08-08"]
비교.round(1)

## 5-3. 시각화 1 — 통영: 주간 원본 vs 12주 이동평균

한 지역만 놓고 "이동평균이 무엇을 하는지" 보여 주는 그림이다.
옅은 선이 원본(노이즈 포함), 진한 선이 12주 이동평균(추세)이다.

In [ ]:
COLORS = {"통영": "#3B5BDB", "거제": "#E8590C", "남해": "#099268"}

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(df.index, df["통영"], color=COLORS["통영"], alpha=0.28, linewidth=1.2, label="주간 원본")
ax.plot(ma12.index, ma12["통영"], color=COLORS["통영"], linewidth=2.2, label="12주 이동평균")

ax.set_title("통영 여행 검색 관심도 — 주간 원본과 12주 이동평균 (2021~2025)")
ax.set_ylabel("검색 관심도 (기간 내 최댓값=100)")
ax.set_xlabel("")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)          # 눈금선은 흐리게: 데이터가 주인공
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)  # 위·오른쪽 테두리 제거

fig.tight_layout()
fig.savefig("images/01_통영_이동평균.png", dpi=150)
plt.show()

## 5-4. 시각화 2 — 세 지역 12주 이동평균 비교

원본 3개를 겹쳐 그리면 선이 엉켜 읽히지 않는다. 이동평균만 그려 흐름을 비교한다.
선 끝에 지역 이름을 직접 붙여, 색만으로 구분하지 않아도 되게 한다.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
for region in df.columns:
    ax.plot(ma12.index, ma12[region], color=COLORS[region], linewidth=2.2, label=region)

# 선 끝에 지역 이름을 직접 붙인다. 끝값이 비슷하면 글자가 겹치므로,
# 값이 낮은 쪽부터 훑으면서 최소 간격(세로 폭의 4%)만큼 벌려 놓는다.
마지막 = ma12.dropna().iloc[-1].sort_values()
최소간격 = (ma12.max().max() - ma12.min().min()) * 0.04
직전y = None
for region, 값 in 마지막.items():
    y = 값 if 직전y is None else max(값, 직전y + 최소간격)
    ax.annotate(region, xy=(ma12.dropna().index[-1], y),
                xytext=(6, 0), textcoords="offset points",
                color=COLORS[region], fontweight="bold", va="center")
    직전y = y

ax.set_title("남해안 3개 여행지 검색 관심도 — 12주 이동평균 비교")
ax.set_ylabel("검색 관심도 (기간 내 최댓값=100)")
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.grid(axis="y", alpha=0.25)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

fig.tight_layout()
fig.savefig("images/02_3개지역_이동평균.png", dpi=150)
plt.show()


## 5-5. 관찰 메모 (숫자로 확인)

그래프에서 눈으로 본 것을 숫자로 확인해 둔다. 해석(왜 그런가)은 단계 7에서 따로 쓴다.

주의: 아래 표의 **2020년 행은 주가 1개뿐**이다(경계 주 2020-12-28). 연도 비교에서는 제외하고 읽는다.

In [ ]:
연도별 = df.groupby(df.index.year).mean().round(1)
연도별.index.name = "연도"
연도별